In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import rasterio
from shapely.geometry import LineString, Point
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx; import basemaps
import pandas as pd
import h3

In [ ]:
# Load the study area polygon
gdf = gpd.read_file("./data/Turku_region_boundary.geojson")  # or .shp

# Make sure it’s in WGS84 (lat/lon) for OSMnx
gdf = gdf.to_crs(epsg=4326)


In [ ]:
import osmnx as ox
import geopandas as gpd


# Extract the geometry
polygon = gdf.unary_union  # or gdf.geometry.iloc[0] if only one feature

default_access = '["access"!~"private"]'  # example placeholder

# Custom OSM filter for the cycling network
custom_bike_filter = (
    f'["highway"]["area"!~"yes"]{default_access}'
    f'["highway"!~"abandoned|bus_guideway|corridor|elevator|'
    f'escalator|motor|no|planned|platform|proposed|raceway|razed|'
    f'rest_area|services|steps"]'
    f'["service"!~"private"]'
    f'["bicycle"!~"no"]'
)

# Download the bike network within the polygon
G = ox.graph_from_polygon(polygon, custom_filter=custom_bike_filter, network_type=None)

In [ ]:
G = ox.truncate.largest_component(G, strongly=False)

In [ ]:
G = ox.add_edge_speeds(G)
G = ox.distance.add_edge_lengths(G)

In [ ]:
# Convert to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)

In [ ]:
# Reproject to Web Mercator (EPSG:3857) for web tiles
edges_web = edges.to_crs(epsg=3857)

In [ ]:
# Reproject to Web Mercator (EPSG:3857) for web tiles
edges_web = edges.to_crs(epsg=3857)

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
edges_web.plot(ax=ax, linewidth=1, color='blue', alpha=0.7)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Remove axes
ax.set_axis_off()
plt.title("Turku Bike Network with Basemap")
plt.tight_layout()
plt.show()

In [ ]:
60.448196, 22.244006 #city center turku
60.441366, 22.295857 # turku other side of the river


# Origin and destination (latitude, longitude) — sample points
origin = (60.448196, 22.244006)      
destination = (60.441366, 22.295857) # Near 


origin_node = ox.distance.nearest_nodes(G, origin[1], origin[0])
dest_node = ox.distance.nearest_nodes(G, destination[1], destination[0])

route = nx.shortest_path(G, origin_node, dest_node, weight='custom_weight')

In [ ]:
ox.plot_graph_route(G, route, route_linewidth=4, node_size=0, bgcolor='white')

In [ ]:
total_length = 0
for u, v in zip(route[:-1], route[1:]):
    # In case of multiple edges between u and v (e.g., parallel paths)
    data = G.get_edge_data(u, v)
    if isinstance(data, dict):
        # Use the first edge (key = 0) or the shortest if needed
        edge = data[min(data)]  # safest: min key
        total_length += edge.get('length', 0)

print(f"Total route length: {total_length:.2f} meters")

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

input_tif = "scratch/cropped_dem_turku.tif"
output_tif = "scratch/cropped_dem_turku_utm32635.tif"
dst_crs = "EPSG:32635"

with rasterio.open(input_tif) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    with rasterio.open(output_tif, "w", **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear,
            )

print("Raster reprojected to EPSG:32635 at", output_tif)

In [ ]:
# Load the graph in the same projection (EPSG:32635)
G_proj = ox.project_graph(G, to_crs="EPSG:32635")

In [ ]:
# --- previous steps ---
dem_path = "scratch/cropped_dem_turku_utm32635.tif"
G_elev = ox.add_node_elevations_raster(G_proj, filepath=dem_path, band=1, cpus=4)


# 1. Check missing elevations
missing_nodes = [
    n for n, data in G_elev.nodes(data=True)
    if data.get('elevation') is None or np.isnan(data.get('elevation'))
]

print(f"Nodes missing elevation before patch: {len(missing_nodes)}")

# 2. Patch missing elevations with 2 meters
patched_count = 0
for n in missing_nodes:
    G_elev.nodes[n]['elevation'] = 2
    patched_count += 1

print(f"Patched {patched_count} nodes with elevation=2")

# 3. Recompute edge grades (slopes) now that all nodes have elevations
G_elev = ox.add_edge_grades(G_elev)

# 4. Verify no more missing elevations
missing_after = [
    n for n, data in G_elev.nodes(data=True)
    if data.get('elevation') is None or np.isnan(data.get('elevation'))
]

print(f"Nodes missing elevation after patch: {len(missing_after)}")

In [ ]:
node_elev = [data["elevation"] for _, data in G_elev.nodes(data=True)]

fig, ax = ox.plot_graph(
    G_elev,
    node_color=node_elev,
    node_size=5,
    #node_cmap="terrain",
    edge_color="gray",
    edge_linewidth=0.5,
)

```python
import geopandas as gpd
from shapely.geometry import LineString, Point

# 1. Nodes GeoDataFrame
nodes_gdf = gpd.GeoDataFrame(
    [{'node': n, 'elevation': data.get('elevation'), 'geometry': Point(data['x'], data['y'])} 
     for n, data in G_elev.nodes(data=True)],
    crs=G_proj.graph['crs']
)

# 2. Filter nodes inside hexes
#nodes_in_hex = nodes_gdf[nodes_gdf.within(hex_gdf.unary_union)]

# 3. Edges GeoDataFrame (only edges where both ends are inside hexes)
edges_gdf = gpd.GeoDataFrame(
    [{'u': u, 'v': v, 'geometry': LineString([Point(G_elev.nodes[u]['x'], G_elev.nodes[u]['y']),
                                             Point(G_elev.nodes[v]['x'], G_elev.nodes[v]['y'])])}
     for u, v, data in G_elev.edges(data=True)
     if u in nodes_in_hex['node'].values and v in nodes_in_hex['node'].values],
    crs=G_proj.graph['crs']
)

# 4. Interactive map with nodes + edges
m = edges_gdf.explore(color='gray')                  # edges
nodes_in_hex.explore(m=m, column='elevation', cmap='Reds', tooltip=['node','elevation'], popup=True)
```

In [ ]:
# Compute edge slopes (grades)
G_slope = ox.add_edge_grades(G_elev)

In [ ]:
# Extract edge grades (slopes) from the graph edges
edges = ox.graph_to_gdfs(G_slope, nodes=False, edges=True)

# Plot the edges colored by slope
fig, ax = plt.subplots(figsize=(12, 12))
edges.plot(
    ax=ax,
    column='grade',          # The slope attribute
    cmap='coolwarm',         # Color map: blue = low, red = high slope
    linewidth=1,
    legend=True,
    legend_kwds={'label': "Slope (grade)", 'shrink': 0.5}
)
ax.set_title("Edge slopes (grades) on the graph")
ax.set_axis_off()
plt.show()

In [ ]:
# Access nodes as a GeoDataFrame
nodes = ox.graph_to_gdfs(G_slope, edges=False)

# Check basic stats on elevation
print(nodes['elevation'].describe())

In [ ]:
import matplotlib.pyplot as plt

edges_gdf = ox.graph_to_gdfs(G_slope, nodes=False, edges=True)
edges_gdf['grade'].hist(bins=50)
plt.xlabel('Slope grade')
plt.ylabel('Count of edges')
plt.title('Distribution of edge grades (slopes)')
plt.show()

print(edges_gdf['grade'].describe())


In [ ]:

edges_gdf = edges_gdf[(edges_gdf["grade"] < 0.1) | (edges_gdf["grade"] > -0.1)]

In [ ]:
edges_gdf.head()

In [ ]:
edges_gdf["grade_pct"] = edges_gdf["grade"] * 100

fig, ax = plt.subplots(figsize=(12, 12))
edges_gdf.plot(
    ax=ax,
    column='grade_pct',
    cmap='coolwarm',
    linewidth=1,
    legend=True,
    legend_kwds={'label': "Slope (%)", 'shrink': 0.5},
    vmin=-25,   # set min color range
    vmax=25     # set max color range
)
ax.set_title("Edge slopes (%) on the graph")
ax.set_axis_off()
plt.show()

In [ ]:
travel_time_matrix = pd.read_csv("./data/travel_time_matrix_bicycle_turku.csv")

In [ ]:
travel_time_matrix.shape

In [ ]:
turku_pois = pd.read_parquet("./data/pois_per_hex_new_class_turku.parquet")

In [ ]:
turku_pois_unique = turku_pois.drop_duplicates(subset="h3_id").copy()

In [ ]:
df_unique_from_ids = travel_time_matrix[['from_id']].drop_duplicates().reset_index(drop=True)

In [ ]:
# Cartesian product
df_combinations = pd.merge(df_unique_from_ids.assign(key=1),
                           turku_pois_unique[['h3_id']].assign(key=1),
                           on='key').drop('key', axis=1)

# Optional: rename for clarity
df_combinations = df_combinations.rename(columns={'h3_id': 'to_id'})

In [ ]:
df_combinations

In [ ]:
travel_time_matrix = df_combinations.copy()

In [ ]:
# travel_time_matrix is the DataFrame to update
travel_time_matrix = travel_time_matrix.dropna()

# Reset the index afterwards
travel_time_matrix.reset_index(drop=True, inplace=True)

In [ ]:
travel_time_matrix.shape

In [ ]:
import h3

# Add lat/lon columns for from_id and to_id
travel_time_matrix['from_lat'], travel_time_matrix['from_lon'] = zip(*travel_time_matrix['from_id'].map(h3.h3_to_geo))
travel_time_matrix['to_lat'], travel_time_matrix['to_lon'] = zip(*travel_time_matrix['to_id'].map(h3.h3_to_geo))

In [ ]:
travel_time_matrix.head()

In [ ]:
# Get all edge attribute keys in G_slope
edge_attributes = set()
for _, _, data in G_slope.edges(data=True):
    edge_attributes.update(data.keys())

print(edge_attributes)# Get all edge attribute keys in G_slope
edge_attributes = set()
for _, _, data in G_slope.edges(data=True):
    edge_attributes.update(data.keys())

print(edge_attributes)

In [ ]:
# Check how many edges have extreme slopes (> 50% or < -50%)
extreme_edges = [
    (u, v, k, data.get("grade", 0.0))
    for u, v, k, data in G_slope.edges(keys=True, data=True)
    if abs(data.get("grade", 0.0)) > 0.5
]

print(f"Number of edges with slope > 50% or < -50%: {len(extreme_edges)}")

# Optionally inspect first few
extreme_edges[:5]

In [ ]:
# Cap extreme slopes to ±0.5
for u, v, k, data in G_slope.edges(keys=True, data=True):
    slope = data.get("grade", 0.0)
    if slope > 0.5:
        data["grade"] = 0.5
    elif slope < -0.5:
        data["grade"] = -0.5

In [ ]:
def slope_penalty(slope):
    # slope expected as fraction (0.05 = 5% grade)
    slope = max(min(slope, 0.3), -0.3)  # cap extreme values for stability
    if slope > 0:       # Uphill: more effort
        return slope
    else:               # Downhill: slight benefit
        return slope * 0.3

for u, v, k, data in G_slope.edges(keys=True, data=True):
    slope = data.get("grade", 0.0)

    # Handle NaN slopes → set to 0
    if slope is None or pd.isna(slope):
        slope = 0.0

    # Convert from % to fraction if necessary
    if abs(slope) > 1.5:
        slope = slope / 100.0

    length = data.get("length", 0.0)
    data["custom_weight"] = length * (1 + slope_penalty(slope))

In [ ]:
## From points to osm id

In [ ]:
# 1. Create geometry for 'from_id' and 'to_id' columns (H3 centroids)
travel_time_matrix["from_geom"] = travel_time_matrix["from_id"].apply(
    lambda h: Point(h3.h3_to_geo(h)[1], h3.h3_to_geo(h)[0])
)
travel_time_matrix["to_geom"] = travel_time_matrix["to_id"].apply(
    lambda h: Point(h3.h3_to_geo(h)[1], h3.h3_to_geo(h)[0])
)

# 2. Convert to GeoDataFrames for convenience
gdf_from = gpd.GeoDataFrame(travel_time_matrix, geometry="from_geom", crs="EPSG:4326")
gdf_to = gpd.GeoDataFrame(travel_time_matrix, geometry="to_geom", crs="EPSG:4326")


In [ ]:
# Get CRS of G_slope
nodes, _ = ox.graph_to_gdfs(G_slope)
graph_crs = nodes.crs

# Reproject OD geometries to graph CRS
gdf_from_proj = gdf_from.to_crs(graph_crs)
gdf_to_proj = gdf_to.to_crs(graph_crs)

In [ ]:
# Get all edge attribute keys in G_slope
edge_attributes = set()
for _, _, data in G_slope.edges(data=True):
    edge_attributes.update(data.keys())

print(edge_attributes)

In [ ]:
import time 

# Convert to NumPy arrays for speed
x_from = gdf_from_proj.geometry.x.to_numpy()
y_from = gdf_from_proj.geometry.y.to_numpy()
x_to   = gdf_to_proj.geometry.x.to_numpy()
y_to   = gdf_to_proj.geometry.y.to_numpy()

# Time it
start_time = time.time()


travel_time_matrix["orig_node"] = ox.distance.nearest_nodes(G_slope, x_from, y_from)
travel_time_matrix["dest_node"] = ox.distance.nearest_nodes(G_slope, x_to, y_to)

elapsed = time.time() - start_time
print(f"Mapping 12M OD pairs took {elapsed:.2f} seconds")

In [ ]:
travel_time_matrix.head()

In [ ]:
travel_time_matrix[['from_id', 'to_id', 'from_lat', 'from_lon', 'to_lat', 'to_lon', 'orig_node', 'dest_node']].to_parquet("./data/OD_cycling_33M_snap_1.parquet")

In [ ]:
travel_time_matrix["from_geom"] = travel_time_matrix["from_geom"].astype(str)

In [ ]:
travel_time_matrix["to_geom"] = travel_time_matrix["to_geom"].astype(str)

In [ ]:
travel_time_matrix.to_parquet("./data/OD_cycling_33M_snap_1_turku.parquet")

In [ ]:
#travel_time_matrix = pd.read_parquet("./data/OD_cycling_33M_snap_1.parquet")

In [ ]:
travel_time_matrix[['from_id', 'to_id', 'from_lat', 'from_lon', 'to_lat', 'to_lon', 'orig_node', 'dest_node']].to_parquet("./data/OD_cycling_33M_snap.parquet")

In [ ]:
node_coords = {node: (data["x"], data["y"]) for node, data in G_slope.nodes(data=True)}


In [ ]:
# Build a GeoDataFrame with the projected nodes
nodes_gdf = gpd.GeoDataFrame(
    {"node": list(node_coords.keys())},
    geometry=[Point(xy) for xy in node_coords.values()],
    crs=G_slope.graph["crs"]  # graph CRS
)

# Reproyectar a WGS84
nodes_wgs84 = nodes_gdf.to_crs(epsg=4326)

# Build a new dict {node: (lon, lat)}
node_coords_wgs84 = dict(zip(
    nodes_wgs84["node"],
    [(geom.x, geom.y) for geom in nodes_wgs84.geometry]
))

In [ ]:
import pandas as pd
import numpy as np

# assuming travel_time_matrix has columns: orig_node, dest_node, from_lon, from_lat, to_lon, to_lat
# node_coords_wgs84 is a dict: {node_id: (lon, lat)}

# filter orig nodes that exist
valid_orig_nodes = [n for n in travel_time_matrix["orig_node"] if n in node_coords_wgs84]
valid_dest_nodes = [n for n in travel_time_matrix["dest_node"] if n in node_coords_wgs84]

# create DataFrame with coordinates for valid orig nodes
df_orig_nodes = pd.DataFrame({
    "node": valid_orig_nodes,
    "lon": [node_coords_wgs84[n][0] for n in valid_orig_nodes],
    "lat": [node_coords_wgs84[n][1] for n in valid_orig_nodes]
})

# create DataFrame with coordinates for valid dest nodes
df_dest_nodes = pd.DataFrame({
    "node": valid_dest_nodes,
    "lon": [node_coords_wgs84[n][0] for n in valid_dest_nodes],
    "lat": [node_coords_wgs84[n][1] for n in valid_dest_nodes]
})

# combine them (optional, remove duplicates)
df_nodes_all = pd.concat([df_orig_nodes, df_dest_nodes]).drop_duplicates(subset="node").reset_index(drop=True)
df_nodes_all


In [ ]:
# rows where orig_node or dest_node is missing in node_coords_wgs84
problem_rows = travel_time_matrix[
    (~travel_time_matrix["orig_node"].isin(node_coords_wgs84))
]


In [ ]:
import numpy as np
from pyproj import Geod

geod = Geod(ellps="WGS84")

# helper function to get node coordinates safely
def safe_coords(n):
    if n in node_coords_wgs84:
        return node_coords_wgs84[n]
    else:
        return (np.nan, np.nan)  # placeholder to compute a default distance later

# get origin coordinates safely
orig_coords = np.array([safe_coords(n) for n in travel_time_matrix["orig_node"]])
dest_coords = np.array([safe_coords(n) for n in travel_time_matrix["dest_node"]])

# compute distances
_, _, dist_orig = geod.inv(
    travel_time_matrix["from_lon"].to_numpy(),
    travel_time_matrix["from_lat"].to_numpy(),
    orig_coords[:,0],
    orig_coords[:,1]
)

_, _, dist_dest = geod.inv(
    travel_time_matrix["to_lon"].to_numpy(),
    travel_time_matrix["to_lat"].to_numpy(),
    dest_coords[:,0],
    dest_coords[:,1]
)

# replace distances where coordinates were missing with 5000
dist_orig = np.where(np.isnan(orig_coords[:,0]), 5000, dist_orig)
dist_dest = np.where(np.isnan(dest_coords[:,0]), 5000, dist_dest)


In [ ]:
travel_time_matrix

In [ ]:
# Store in the DataFrame
travel_time_matrix["dist_orig_m"] = dist_orig
travel_time_matrix["dist_dest_m"] = dist_dest

# Definir un umbral (ejemplo: 1000 m)
threshold = 500  

# Detect problematic rows
mask_bad = (travel_time_matrix["dist_orig_m"] > threshold) | (travel_time_matrix["dist_dest_m"] > threshold)

# Subset of problematic rows
bad_matches = travel_time_matrix[mask_bad]

print(f"Warning: {mask_bad.sum()} rows with distances greater than {threshold} m")

In [ ]:
bad_matches

In [ ]:
travel_time_matrix = travel_time_matrix[(travel_time_matrix["dist_orig_m"] < threshold) & (travel_time_matrix["dist_dest_m"] < threshold)]

In [ ]:

# Extract custom_weight from G_slope edges into a DataFrame
custom_weight_values = [
    data.get("custom_weight", None)
    for _, _, _, data in G_slope.edges(keys=True, data=True)
]

# Convert to Series and describe
pd.Series(custom_weight_values).describe()

In [ ]:
# Check for edges with missing custom_weight
missing_weights = [
    (u, v, k) 
    for u, v, k, data in G_slope.edges(keys=True, data=True) 
    if "custom_weight" not in data
]
print(f"Edges without custom_weight: {len(missing_weights)}")

In [ ]:
neg_or_zero = [
    (u, v, k, data["custom_weight"])
    for u, v, k, data in G_slope.edges(keys=True, data=True)
    if data["custom_weight"] <= 0
]
print(f"Edges with <= 0 weight: {len(neg_or_zero)}")

In [ ]:
import math

nan_edges = []
inf_edges = []

for u, v, k, data in G_slope.edges(keys=True, data=True):
    w = data.get("custom_weight", None)
    if w is None:
        continue
    if isinstance(w, float) and math.isnan(w):
        nan_edges.append((u, v, k))
    elif isinstance(w, float) and not math.isfinite(w):
        inf_edges.append((u, v, k))

print(f"NaN weights: {len(nan_edges)}")
print(f"Infinite weights: {len(inf_edges)}")

# Optionally inspect first few
print("Example NaN edges:", nan_edges[:5])
print("Example Inf edges:", inf_edges[:5])

In [ ]:
travel_time_matrix

In [ ]:
# Convert to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G_slope, nodes=True, edges=True)

In [ ]:
# Reproject to Web Mercator (EPSG:3857) for web tiles
edges_web = edges.to_crs(epsg=3857)

# Plot
fig, ax = plt.subplots(figsize=(12, 12))
edges_web.plot(ax=ax, linewidth=1, color='blue', alpha=0.7)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Remove axes
ax.set_axis_off()
plt.title("Helsinki Bike Network with Basemap")
plt.tight_layout()
plt.show()

In [ ]:
sample_df = travel_time_matrix.copy()

In [ ]:
import time
import pandas as pd
import networkx as nx

# 1. Taking all
sample_df_subset = sample_df.copy()

# 2. Prepare timer
start_time = time.time()

results = []

# 3. Loop over unique origins in the subset
for orig in sample_df_subset["orig_node"].unique():
    # Dijkstra distances from this origin
    lengths = nx.single_source_dijkstra_path_length(G_slope, orig, weight="custom_weight")
    
    mask = sample_df_subset["orig_node"] == orig
    subset = sample_df_subset.loc[mask, ["orig_node", "dest_node"]].copy()
    subset["travel_distance"] = subset["dest_node"].map(lengths)
    
    results.append(subset)

# 4. Combine all results
travel_time_matrix_result = pd.concat(results, ignore_index=True)

# 5. Time elapsed
elapsed = time.time() - start_time
print(f"Finished 10M OD pairs in {elapsed:.2f} seconds")


In [ ]:
travel_time_matrix_result

In [ ]:
travel_time_matrix_result.to_parquet("./data/V1_distance_nx_cycling_final_turku.parquet")

In [ ]:
travel_time_matrix_result = pd.read_parquet("./data/V1_distance_nx_cycling_1_turku.parquet")

In [ ]:
travel_time_matrix_result.sort_values("travel_distance")

In [ ]:
travel_time_matrix_result.describe()

In [ ]:
travel_time_matrix_result.head()

In [ ]:
#travel_time_matrix_result = travel_time_matrix_result.merge(
 #   travel_time_matrix[['orig_node', 'dest_node', 'from_id', 'to_id']],
   # on=['orig_node', 'dest_node'],
    #how='inner'
#)

In [ ]:
# 1) Make sure keys are numeric (optional, only if they are not already)
for c in ["orig_node", "dest_node"]:
    travel_time_matrix_result[c] = pd.to_numeric(travel_time_matrix_result[c], errors="raise")
    travel_time_matrix[c]         = pd.to_numeric(travel_time_matrix[c],         errors="raise")

# 2) Build lookup with unique (orig_node, dest_node) and travel_time
lookup = (
    travel_time_matrix_result[["orig_node", "dest_node", "travel_distance"]]
    .drop_duplicates(subset=["orig_node", "dest_node"])
)

# 3) Merge travel_time_matrix with lookup on BOTH keys
merged = travel_time_matrix.merge(
    lookup,
    on=["orig_node", "dest_node"],
    how="left"   # keep all rows from travel_time_matrix
)

# 4) Optional: sanity check for missing travel_time
missing = merged["travel_distance"].isna().sum()
if missing:
    print(f"Warning: {missing} rows in travel_time_matrix had no match in travel_time_matrix_result.")


In [ ]:
travel_time_matrix_result = merged.copy()

In [ ]:
# Bad matches from snap to network
# 1) Build a set of bad (from_id, to_id) pairs
bad_pairs = set(bad_matches[["from_id", "to_id"]].itertuples(index=False, name=None))

# 2) Drop those rows from travel_time_matrix_result
filtered_result = travel_time_matrix_result[
    ~travel_time_matrix_result[["from_id", "to_id"]].apply(tuple, axis=1).isin(bad_pairs)
].copy()

print(f"Removed {len(travel_time_matrix_result) - len(filtered_result)} rows from travel_time_matrix_result")

In [ ]:
travel_time_matrix_result = filtered_result.copy()

In [ ]:
# emission factor in grams CO2 per km
emission_factor = 21  # g/km

# convert meters to km and multiply
travel_time_matrix_result["co2_emissions_g"] = (
    travel_time_matrix_result["travel_distance"] / 1000 * emission_factor
)

In [ ]:
travel_time_matrix_result.sort_values("travel_distance")

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_bike_co2_250 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 250].copy()
df_bike_co2_500 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 500].copy()
df_bike_co2_125 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 125].copy()

In [ ]:
df_bike_co2_2000 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 2000].copy()

In [ ]:
df_bike_co2_3000 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 3000].copy()

In [ ]:
df_bike_co2_3000

In [ ]:
df_bike_co2_3000["from_geom"] = df_bike_co2_3000["from_geom"].astype(str)
df_bike_co2_3000["to_geom"] = df_bike_co2_3000["to_geom"].astype(str)

df_bike_co2_3000.to_parquet("output/bike_co2_3000_turku.parquet")

In [ ]:
df_bike_co2_3000.head()

In [ ]:
turku_pois

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    turku_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_co2_2000_merged = df_bike_co2_2000.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_2000_merged = df_bike_co2_2000_merged.drop(columns=['h3_id'])


# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_2000_merged[col] = df_bike_co2_2000_merged[col].fillna(0) + 0

In [ ]:
#df_bike_co2_2000_merged.to_parquet("./output/bikes_co2_all_2000.parquet")

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    turku_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_co2_250_merged = df_bike_co2_250.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_250_merged = df_bike_co2_250_merged.drop(columns=['h3_id'])


# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_250_merged[col] = df_bike_co2_250_merged[col].fillna(0) + 0

In [ ]:
from shapely.geometry import Polygon
import h3

def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

pois_grouped["geometry"] = pois_grouped["h3_id"].apply(h3_to_polygon)

# 4) Convert to GeoDataFrame
gdf_pois = gpd.GeoDataFrame(pois_grouped, geometry="geometry", crs="EPSG:4326")

In [ ]:
df_bike_co2_250_merged[df_bike_co2_250_merged["from_id"] == df_bike_co2_250_merged["to_id"]]

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Education',
    'Healthcare and Health',
    'Others / Not sure',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]

grouped_summary = df_bike_co2_250_merged.groupby('from_id')[category_cols + ['travel_distance']].agg({
    'Education': 'sum',
    'Healthcare and Health': 'sum',
    'Others / Not sure': 'sum',
    'Recreational, Outdoors': 'sum',
    'Shopping, Errands': 'sum',
    'Social, Cultural': 'sum',
    
}).reset_index()

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Education',
    'Healthcare and Health',
    'Others / Not sure',
    'Recreational, Outdoors',
    'Shopping, Errands',
    'Social, Cultural'
]].sum(axis=1)

In [ ]:
grouped_summary.sort_values(by="Education")

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches


# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary.columns:
    geometry = grouped_summary['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='lower left')

plt.suptitle("POI Categories by Hexagon in 250g trip by bike", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()



In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    hsk_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_co2_125_merged = df_bike_co2_125.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_125_merged = df_bike_co2_125_merged.drop(columns=['h3_id'])


# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_125_merged[col] = df_bike_co2_125_merged[col].fillna(0) + 0

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_bike_co2_125_merged.groupby('from_id')[category_cols + ['travel_distance']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'travel_distance':'mean'
    
}).reset_index()

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary.columns:
    geometry = grouped_summary['from_id'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=7)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='upper left')

plt.suptitle("POI Categories by Hexagon in 125g trip by bike", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()




In [ ]:
### Check examples

In [ ]:
grouped_summary.sort_values(by="Jobs, Professional Services & Religious")

In [ ]:
grouped_summary[grouped_summary["from_id"].isin(["89089969537ffff"])]

In [ ]:
## 891126d1e5bffff LAJAASALO

In [ ]:
laja = df_bike_co2_250_merged[df_bike_co2_250_merged["from_id"].isin(["891126d3303ffff"])].sort_values(by="Well-being & Lifestyle")

In [ ]:
df_bike_co2_250_merged[df_bike_co2_250_merged["from_id"].isin(["891126d3303ffff"])].sort_values(by="Well-being & Lifestyle")

In [ ]:
laja.sum()

In [ ]:
target_ids = ["891126d332bffff", "891126d00abffff"]
df_selected = df_bike_co2_125[df_bike_co2_125["from_id"].isin(target_ids)]


In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon
from h3 import h3

# Step 1: Filter df_selected for the two from_id values
from_ids = ["89089969537ffff"]
df_a = df_bike_co2_125[df_bike_co2_125["from_id"] == from_ids[0]]

# Step 2: Function to convert H3 IDs to polygons
def h3_list_to_gdf(h3_ids):
    polygons = []
    for h in h3_ids:
        # Make sure the ID is a string
        h = str(h)
        # Convert H3 to lat/lon polygon
        boundary = h3.h3_to_geo_boundary(h, geo_json=True)
        poly = Polygon([(lon, lat) for lon, lat in boundary])
        polygons.append(poly)
    return gpd.GeoDataFrame({"h3_id": h3_ids}, geometry=polygons, crs="EPSG:4326")

# Step 3: Convert the to_id codes to polygons for each dataset
gdf_a = h3_list_to_gdf(df_a["to_id"].unique())


# Step 4: Optional - join back original info if needed
gdf_a = gdf_a.merge(df_a, left_on="h3_id", right_on="to_id", how="left")

# Now gdf_a and gdf_b have correct hexagons in Finland


In [ ]:
df_a[df_a["to_id"]=="891126d10d3ffff"]

In [ ]:
gdf_a.explore()

In [ ]:
#891126d10d3ffff Suomenlinna


In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon
from h3 import h3

# Step 1: Filter df_selected for the two from_id values
from_ids = ["891126d1e5bffff"]
df_b = df_bike_co2_125[df_bike_co2_125["from_id"] == from_ids[0]]

# Step 2: Function to convert H3 IDs to polygons
def h3_list_to_gdf(h3_ids):
    polygons = []
    for h in h3_ids:
        # Make sure the ID is a string
        h = str(h)
        # Convert H3 to lat/lon polygon
        boundary = h3.h3_to_geo_boundary(h, geo_json=True)
        poly = Polygon([(lon, lat) for lon, lat in boundary])
        polygons.append(poly)
    return gpd.GeoDataFrame({"h3_id": h3_ids}, geometry=polygons, crs="EPSG:4326")

# Step 3: Convert the to_id codes to polygons for each dataset
gdf_b = h3_list_to_gdf(df_b["to_id"].unique())


# Step 4: Optional - join back original info if needed
gdf_b = gdf_b.merge(df_b, left_on="h3_id", right_on="to_id", how="left")

In [ ]:
# Origin and destination (latitude, longitude) — sample points
origin = (60.171496,	24.958108)      # Near Kruunun
destination = (60.165427,24.943369) # Near center

	
origin_node = ox.distance.nearest_nodes(G, origin[1], origin[0])
dest_node = ox.distance.nearest_nodes(G, destination[1], destination[0])

route = nx.shortest_path(G, origin_node, dest_node, weight='custom_weight')

In [ ]:
df_bike_co2_62 = travel_time_matrix_result[travel_time_matrix_result["co2_emissions_g"] <= 62].copy()

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    hsk_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_bike_co2_62_merged = df_bike_co2_62.merge(pois_grouped, how='left', left_on='to_id', right_on='h3_id')
df_bike_co2_62_merged = df_bike_co2_62_merged.drop(columns=['h3_id'])


# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
    df_bike_co2_62_merged[col] = df_bike_co2_62_merged[col].fillna(0) + 0

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_bike_co2_62_merged.groupby('from_id')[category_cols + ['travel_distance']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'travel_distance':'mean'
    
}).reset_index()

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['from_id'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=8)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()